# Stage 3: A/B Testing Analysis

## 📚 Objectives
- Simulate A/B test experiment with treatment and control groups
- Perform statistical significance testing (Z-test, T-test)
- Calculate key metrics (lift, effect size, p-value)
- Implement statistical functions for production use

## 📋 Agenda
1. Generate synthetic A/B test data
2. Calculate descriptive statistics
3. Perform Z-test for conversion rate
4. Perform T-test for continuous metrics (AOV)
5. Calculate power and sample size
6. Implement functions in src/analysis/ab_test.py

## 🎯 Foundational Concepts: A/B Testing Essentials

### What is A/B Testing?

**A/B Testing** (also called "split testing" or "controlled experiments") is a method to compare two versions of something by:
1. **Randomly splitting** your users into two groups:
   - **Control Group**: Gets the original/baseline experience
   - **Treatment Group**: Gets the new/modified experience
2. **Measuring key metrics** for each group over the same time period
3. **Comparing results** statistically to determine if differences are significant (not due to random chance)

**Real-world examples:**
- E-commerce: Test new checkout flow vs. current checkout
- Social media: Test new feed algorithm vs. old algorithm
- SaaS: Test different pricing tiers vs. current pricing
- Email marketing: Test subject line A vs. subject line B

**Why it matters:** Without randomization and statistical testing, business decisions are based on correlation (X happened, Y changed) rather than causation (X caused Y to change).

---

### What is Conversion Rate?

**Conversion Rate** measures the percentage of users who completed a desired action:

$$\text{Conversion Rate} = \frac{\text{Number of Users Who Converted}}{\text{Total Number of Users}} \times 100\%$$

**Key definitions:**
- **Convert**: Complete the target action (e.g., make a purchase, sign up, click a button)
- **Non-convert**: Don't complete the action (bounce, abandon cart, etc.)

**Example:**
- Control group: 500 users visit page → 50 purchase = **10% conversion rate**
- Treatment group: 500 users visit page → 75 purchase = **15% conversion rate**
- **Lift** (improvement) = (15% - 10%) / 10% = **50% increase**

---

### Understanding Order Value in Raw Data

#### Current State of Raw Kaggle Data
The Olist e-commerce dataset contains **historical transaction records** with:
- `order_value`: Total monetary amount per order (already includes shipping + items)
- `order_status`: Whether order was completed, cancelled, etc.

#### Why We Need to Handle Order Value for A/B Testing

When doing A/B testing, we need to measure **two related but distinct metrics**:

| Metric | Definition | Use Case |
|--------|-----------|----------|
| **Conversion Rate** | % of users who purchased (yes/no) | Did the experiment change purchase behavior? |
| **Average Order Value (AOV)** | Mean dollar amount of orders | Did the experiment change spending among buyers? |

#### How and Why to Update Order Value

**Problem:** Raw data may have:
1. **Incomplete/cancelled orders** → Should we include them in AOV calculation?
2. **Multiple columns** needing aggregation (item price, shipping, discounts)
3. **Inconsistent treatment logic** → Control and treatment groups should use identical data processing

**Solution: Implement consistent order value processing**

| Step | Action | Why |
|------|--------|-----|
| **1. Filter by order_status** | Keep only completed orders (status='delivered') | Only completed purchases represent true revenue |
| **2. Group by user_id** | Sum order values per user | Calculate AOV per user, not per order |
| **3. Handle missing values** | Remove NULL order_value rows | Avoid bias from incomplete records |
| **4. Apply to BOTH groups equally** | Use same logic for control + treatment | Ensures fair comparison (not confounded by data cleaning) |

**Example scenario:**
```
Raw user data:
- User 101: [order1=$50, order2=$30, order3=CANCELLED]
- User 102: [order1=NULL, order2=$80]

After cleaning (completed orders only):
- User 101: Total spend = $80 → AOV = $80 (2 orders)
- User 102: Total spend = $80 → AOV = $80 (1 order)
```

**Impact on analysis:**
- **Without cleaning**: Biases AOV if control/treatment have different cancellation rates
- **With cleaning**: Fair comparison of how the experiment affects spending behavior

---

## 1. Prerequisites: Why Kaggle Data Needs Simulation

### The Core Challenge
- **Kaggle data**: 历史交易记录（一套既定事实）
- **A/B Testing**: 需要对比两个版本的并行试验

**Solution**: We need to synthetically partition users into Control/Treatment and inject treatment effects.

### Hash-based Randomization (Industry Standard)
Instead of random assignment (which changes each time), we use **deterministic hashing**:
- Ensures **same user always sees same variant** (consistency)
- Provides **reproducible 50/50 split** (fairness)
- Works across different system components (web, mobile, etc.)

### Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import norm
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.power import tt_ind_solve_power
import matplotlib.pyplot as plt
import seaborn as sns
import hashlib

# TODO: Set random seed for reproducibility
# np.random.seed(42)
# sns.set_style('whitegrid')

## 2. Generate Synthetic A/B Test Data

**⚠️ Critical Concept:** Why Kaggle datasets don't have A/B test labels

The Olist dataset contains historical transaction records—established facts that already happened. It doesn't tell us "what if we changed the page?" 

**Our task:** Simulate randomization to demonstrate A/B testing capability by:
1. **Splitting users into Control/Treatment using hash-based randomization** (industry standard for consistency)
2. **Injecting uplift** into the treatment group to model the effect of the new page layout

---

### 🔬 Deep Dive: Synthetic Data Generation & Uplift Injection

#### What is Synthetic Data Generation?

**Synthetic data** is artificially generated data that mirrors the statistical properties of real data but is created in code rather than collected from actual users. It does NOT modify the original raw Kaggle dataset.

**Key Point:** Synthetic A/B test data ≠ Modified historical data
- ❌ We do NOT edit the raw Kaggle CSV files
- ❌ We do NOT change historical orders that already happened
- ✅ We CREATE a new dataset that SIMULATES what an A/B test would look like

#### Why Create Synthetic A/B Test Data?

| Reason | Explanation |
|--------|-------------|
| **Historical data isn't experimental** | Kaggle data represents what DID happen, not what COULD happen if we tested something |
| **No baseline for comparison** | Historical data has no "control" and "treatment" groups |
| **Demonstrates capability** | Shows you CAN run A/B tests if you had real experimental data |
| **Reproducibility** | Same code = same results every time (unlike real randomness) |
| **Multiple scenarios** | Can test different uplift assumptions without re-collecting data |
| **Risk-free testing** | No real customers affected, only simulated behavior |

#### How Synthetic Data Generation Works (Step-by-Step)

**Step 1: Assign Users to Groups (Deterministic Hashing)**
```
Raw User ID → Hash Function → Even/Odd → Control or Treatment
User 101   → MD5 hash     → Even  → Control
User 102   → MD5 hash     → Odd   → Treatment  
User 103   → MD5 hash     → Even  → Control
```

**Key Property:** Same user ALWAYS gets same group (deterministic), unlike `random()` which changes each time.

---

**Step 2: Generate Conversion Outcomes with Group-Specific Probabilities**

For each user, simulate: "Did they purchase?"

```python
# CONTROL GROUP: Baseline behavior (historical average)
baseline_conversion_rate = 0.10  # 10% of users purchase
if random(0, 1) < 0.10:  # Random coin flip
    is_converted = 1      # Yes, purchased
else:
    is_converted = 0      # No, didn't purchase

# TREATMENT GROUP: Enhanced behavior (simulating new page layout)
treatment_conversion_rate = 0.12  # 12% of users purchase (2% uplift)
if random(0, 1) < 0.12:  # Same coin flip, but higher probability
    is_converted = 1      # Yes, purchased
else:
    is_converted = 0      # No, didn't purchase
```

**Why different probabilities?** We're SIMULATING the effect of the new page layout by giving treatment users a higher chance of converting.

---

**Step 3: Generate Order Values for Converters**

For users who purchased, simulate: "How much did they spend?"

```python
# CONTROL GROUP: Baseline order value distribution
if is_converted == 1:
    order_value = sample from Normal(mean=$85.50, std=$15)
    # Example: could be $82, $88, $95, etc.

# TREATMENT GROUP: Higher order value distribution
if is_converted == 1:
    order_value = sample from Normal(mean=$92.30, std=$15)  # +$6.80 uplift
    # Example: could be $89, $95, $102, etc.
```

**Why different distributions?** We're simulating that the new page layout also encourages higher spending.

---

#### Example: Building a Synthetic Dataset

**Input (from 10,000 users):**
```
User_ID | Kaggle Raw Data (unchanged)
1       | [historical orders from CSV]
2       | [historical orders from CSV]
3       | [historical orders from CSV]
...
```

**Process (synthetic generation):**
```python
For each user_id in 1..10000:
  1. variant_group = hash_based_assignment(user_id)
     → User 1: Control, User 2: Treatment, User 3: Control, ...
  
  2. conversion_prob = 0.10 if Control else 0.12
     → Control: 10% chance, Treatment: 12% chance
  
  3. is_converted = flip_coin(conversion_prob)
     → User 1: converted=0, User 2: converted=1, User 3: converted=1, ...
  
  4. if is_converted:
       order_value = random_normal(mean, std)
       → User 2: $89.50 (treatment mean), User 3: $87.20 (control mean)
     else:
       order_value = NULL
```

**Output (synthetic A/B test dataset):**
```
User_ID | Variant_Group | is_converted | order_value
1       | Control       | 0            | NULL
2       | Treatment     | 1            | $89.50      ← synthetic!
3       | Control       | 1            | $87.20      ← synthetic!
4       | Treatment     | 0            | NULL
5       | Control       | 1            | $92.15      ← synthetic!
...
```

---

#### What We're NOT Doing (Common Misconception)

❌ **Wrong:** "Manually go through Kaggle CSV and change some $50 orders to $80"
- This would corrupt historical data
- This would be obvious manipulation
- This wouldn't reflect realistic behavior

✅ **Right:** "Use probability distributions to generate NEW order values for NEW simulated transactions"
- Historical data stays unchanged
- Simulated behavior is based on statistical distributions
- Realistic simulation of what COULD happen

---

#### Why Uplift Values Matter

We inject **realistic uplift assumptions** based on marketing/product intuition:

| Metric | Control (Baseline) | Treatment (Uplifted) | Why |
|--------|-------------------|----------------------|-----|
| **Conversion Rate** | 10% | 12% (+2%) | New page layout makes it easier to checkout |
| **AOV** | $85.50 | $92.30 (+$6.80) | Customers spend more when less friction exists |

**These are ASSUMPTIONS** — in a real A/B test, actual results might be different. But using realistic assumptions makes the simulation valuable for:
- **Testing our analytics pipeline** (does our code correctly detect differences?)
- **Understanding sample size needs** (do 5,000 users per group have enough power?)
- **Planning real experiments** (what effect size should we expect?)

---

## 1.5️⃣  Process Raw Kaggle Data

Click the cell below to load and transform real Olist data instead of generating synthetic data.

In [ ]:
# OPTION: Transform Real Kaggle Olist Data (if downloaded)

# TODO: If using real Kaggle data, implement this template:
# 1. Load CSV files: olist_orders_dataset.csv, olist_order_items_dataset.csv
# 2. Assign each customer to Control/Treatment using get_group()
# 3. Calculate order values (sum of items per order)
# 4. Inject treatment effect (+2% conversion uplift, +$6.80 AOV uplift)
# 5. Create analysis dataframe with columns: user_id, variant_group, is_converted, order_value

# Example structure:
# orders_df['variant_group'] = orders_df['customer_id'].apply(get_group)
# For treatment group, multiply order_value by 1.08 (8% uplift)

# For now, we'll proceed with synthetic data generation below
print("Template provided for real Kaggle data transformation")

In [ ]:
# TODO: Implement hash-based user group assignment function
# Key requirements:
# - Take user_id as input
# - Use MD5 hash to ensure deterministic assignment (same user = same group)
# - Return 'Control' or 'Treatment' (50/50 split)
# - Hint: Use hashlib.md5() and modulo operator

def get_group(user_id):
    """
    Assign user to Control or Treatment group based on user_id hash.
    This ensures the SAME user always sees the SAME variant.
    
    Args:
        user_id: Unique user identifier
        
    Returns:
        'Control' or 'Treatment' (50/50 split)
    """
    # TODO: Implement hash-based randomization
    pass


# TODO: Create user dataframe with group assignments
# Option A: Use extracted user_ids from Kaggle data
# Option B: Generate synthetic user data (e.g., 10,000 users)

# n_total_users = 10000
# user_ids = np.arange(1, n_total_users + 1)
# 
# TODO: Create users_df with columns: user_id, variant_group
# TODO: Print distribution of users across groups

print("TODO: User assignment and distribution")



### 📌 Implementation Task: Hash-based User Assignment

**After testing the `get_group()` function in this notebook, implement in:** `src/analysis/ab_test.py`

Function to implement:
```python
def get_group(user_id):
    """
    Assign user to Control or Treatment group based on deterministic hash.
    
    Args:
        user_id: Unique user identifier (int or string)
        
    Returns:
        'Control' or 'Treatment'
    """
    # TODO: Use hashlib.md5() to hash the user_id
    # TODO: Convert hash to integer and use modulo 2
    # TODO: Return 'Control' if (hash % 2 == 0), 'Treatment' otherwise
```

**Key benefit:** Same user will ALWAYS be assigned to the same group (deterministic), unlike random() which changes each time.

In [ ]:
# TODO: Step 2: Simulate behavior with effect injection
# Requirements:
# - Baseline conversion rate: 10% (control group)
# - Baseline AOV: $85.50 (control group)
# - Treatment uplift: +2% conversion, +$6.80 AOV
# 
# For each user:
# 1. Determine conversion probability based on their group
# 2. Randomly generate whether they converted (use np.random.rand())
# 3. If converted, generate their order value (use np.random.normal())
# 4. Create dataframe with columns: user_id, variant_group, is_converted, order_value

# baseline_conversion_rate = 0.10
# baseline_aov = 85.50
# conversion_uplift = 0.02
# aov_uplift = 6.80

# TODO: Generate individual-level behavior for each user
# TODO: Create ab_test_df with all user records

print("TODO: Simulate A/B test behavior with effect injection")

## 3. Descriptive Statistics

In [ ]:
# TODO: Calculate descriptive statistics by group
# Requirements:
# 1. Group ab_test_df by variant_group
# 2. Calculate for each group:
#    - Sum of conversions
#    - Total count
#    - Mean conversion rate
#    - Mean order value (for converted users only)
#    - Std of order value
#
# 3. Extract key metrics:
#    - control_conversion_rate
#    - treatment_conversion_rate
#    - control_aov (mean order value for converted control users)
#    - treatment_aov (mean order value for converted treatment users)
#
# 4. Display summary table with clear formatting

# Expected output format:
# | Group      | Conversions | Conv. Rate | Avg AOV  |
# |------------|-------------|-----------|----------|
# | Control    | XXX / 5000  | 10.00%    | $85.50   |
# | Treatment  | XXX / 5000  | 12.00%    | $92.30   |

print("TODO: Calculate and display summary statistics")

## 4. Z-Test for Conversion Rate

### 🔬 Statistical Testing Concepts

#### What is the Z-Test?

**Definition:** The Z-Test is a statistical test that compares **two proportions** (percentages) to determine if the difference between them is statistically significant or just due to random chance.

**When to use it:** When you have binary/categorical data:
- ✅ Conversions: Did they purchase (yes/no)?
- ✅ Click-through rate: Did they click the button (yes/no)?
- ✅ Sign-up rate: Did they sign up (yes/no)?

**The hypothesis:**
- **Null Hypothesis (H₀):** Control conversion rate = Treatment conversion rate (no difference)
- **Alternative Hypothesis (H₁):** Control conversion rate ≠ Treatment conversion rate (there IS a difference)

**How it works:**
1. Calculate conversion rate for each group
2. Compute the standard error (how much variation we'd expect by chance)
3. Calculate the Z-statistic: $Z = \frac{p_{treatment} - p_{control}}{SE}$
4. Compare to a threshold (critical value for α=0.05) to get the p-value

**Example:**
```
Control group:    500 users, 50 converted = 10% conversion rate
Treatment group:  500 users, 60 converted = 12% conversion rate
Difference:       2% (absolute), 20% (relative)

Z-Test Result:
  Z-statistic = 1.45
  p-value = 0.074
  Conclusion: NOT statistically significant (p > 0.05)
  
Interpretation: There's a 7.4% chance we'd see this 2% difference 
just by random chance. Since 7.4% > 5%, we can't confidently 
say the treatment works better.
```

**Why it's useful for A/B testing:**
- Tests the PRIMARY metric: whether the experiment changes conversion behavior
- Gives a p-value: probability that the difference is due to random chance
- Quickly answers: "Is this improvement real or just luck?"

In [ ]:
# TODO: Z-Test for Conversion Rate Comparison
# Hypothesis test:
# - H0: p_control = p_treatment (no difference)
# - H1: p_control ≠ p_treatment (two-tailed test)
# - Significance level: α = 0.05
#
# Requirements:
# 1. Count conversions in each group
# 2. Use proportions_ztest() from statsmodels
# 3. Extract z_stat and p_value
# 4. Determine if result is significant (p_value < 0.05)
# 5. Display results with clear formatting
#
# Expected output:
# Z-statistic: X.XXXX
# P-value: 0.XXXX
# Significant at α=0.05: YES / NO

# Hint: proportions_ztest(count, nobs, alternative='two-sided')
# where count = [control_successes, treatment_successes]
#       nobs = [control_total, treatment_total]

print("TODO: Perform two-proportion Z-test on conversion rates")

## 5. T-Test for AOV (Continuous Metric)

### What is the T-Test?

**Definition:** The T-Test is a statistical test that compares **two means** (averages) to determine if the difference is statistically significant.

**When to use it:** When you have continuous/numerical data:
- ✅ Average Order Value (AOV): How much do customers spend on average?
- ✅ Average time on page: How long do users stay?
- ✅ Average cart value: What's the typical cart total?

**The hypothesis:**
- **Null Hypothesis (H₀):** Mean AOV in Control = Mean AOV in Treatment (no difference)
- **Alternative Hypothesis (H₁):** Mean AOV in Control ≠ Mean AOV in Treatment (there IS a difference)

**How it works:**
1. Calculate average order value for each group
2. Calculate the standard deviation (spread of values) for each group
3. Compute the T-statistic: $t = \frac{\bar{x}_{treatment} - \bar{x}_{control}}{SE}$
4. Compare to a threshold based on sample size to get the p-value

**Example:**
```
Control group:    5,000 users, Mean AOV = $85.50, Std = $15
Treatment group:  5,000 users, Mean AOV = $92.30, Std = $15
Difference:       $6.80 (absolute), 8% (relative)

T-Test Result:
  T-statistic = 12.34
  p-value = 0.0001
  Conclusion: HIGHLY statistically significant (p < 0.001)
  
Interpretation: There's only a 0.01% chance we'd see this $6.80 
difference just by random chance. The treatment really does 
increase order value!
```

**Why it's useful for A/B testing:**
- Tests SECONDARY metrics: whether experiment changes spending behavior
- Works with normal distributions of continuous data
- Provides confidence that improvement is real, not random

---

#### What is Cohen's d (Effect Size)?

**Definition:** Cohen's d measures the **magnitude** of the difference between two groups, independent of sample size.

**Formula:** $d = \frac{\bar{x}_{treatment} - \bar{x}_{control}}{\text{pooled\_std}}$

**Interpretation:**
- $d = 0.2$: Small effect
- $d = 0.5$: Medium effect
- $d = 0.8$: Large effect

**Example:**
```
Two scenarios with same p-value = 0.05:
  Scenario A: $85 → $86 difference, Cohen's d = 0.07 (tiny!)
  Scenario B: $85 → $95 difference, Cohen's d = 0.67 (substantial!)
  
Both reject H₀, but Scenario B has MUCH MORE business impact.
```

**Why it matters:** P-value alone doesn't tell you the size of the effect. With large samples, tiny differences become "significant" even if they're not meaningful.

---

#### Important: Filter for Converters Only!

**Key difference:**
- **Z-Test:** Binary outcomes (converted or not) → Proportions → Z-distribution
- **T-Test:** Continuous outcomes (order values) → Means → T-distribution

**In our context:**
- We already tested **if** treatment increases conversions (Z-test)
- Now we test **by how much** treatment increases order value (T-test)

**Data prep requirement:**
```python
# CORRECT: Only compare order values for customers who actually purchased
control_aov = control_df[control_df['is_converted'] == 1]['order_value']
treatment_aov = treatment_df[treatment_df['is_converted'] == 1]['order_value']
# → Sample sizes: 500 (control converters), 600 (treatment converters)

# WRONG: Include non-converters (missing order values)
control_aov = control_df['order_value']  # Has NULLs!
# → Biased results
```

**Expected outcome:** If treatment BOTH increases conversions AND increases AOV, the business impact is multiplicative!

---

In [ ]:
# TODO: T-Test for AOV (Average Order Value) Comparison
# Hypothesis test:
# - H0: mean_aov_control = mean_aov_treatment
# - H1: mean_aov_control ≠ mean_aov_treatment
# - Significance level: α = 0.05
#
# Requirements:
# 1. Filter only converted users (is_converted == 1) from both groups
# 2. Use scipy.stats.ttest_ind() for independent samples t-test
# 3. Calculate Cohen's d effect size: (mean_treatment - mean_control) / pooled_std
# 4. Extract t_stat and p_value
# 5. Display results with effect size
#
# Expected output:
# T-statistic: X.XXXX
# P-value: 0.XXXX
# Cohen's d: 0.XXXX
# Significant at α=0.05: YES / NO
#
# Formulas:
# pooled_std = sqrt((std_control^2 + std_treatment^2) / 2)
# Cohen's d = (mean_treatment - mean_control) / pooled_std

print("TODO: Perform independent samples T-test on AOV")

## 6. Calculate Lift and Effect Size

In [ ]:
# TODO: Calculate business metrics and interpret results
# Requirements:
# 1. Calculate conversion rate lift (absolute and relative):
#    - Absolute: treatment_rate - control_rate
#    - Relative: (treatment_rate - control_rate) / control_rate
#
# 2. Calculate AOV lift (absolute and relative):
#    - Absolute: treatment_aov - control_aov
#    - Relative: (treatment_aov - control_aov) / control_aov
#
# 3. Project business impact if rolled out to all treatment users:
#    - Additional conversions: n_treatment * conversion_absolute_lift
#    - Incremental revenue: additional_conversions * treatment_aov
#
# 4. Display in clear, business-friendly format
#
# Expected output:
# CONVERSION RATE LIFT:
#    Absolute Lift: 2.00% (e.g., 10% → 12%)
#    Relative Lift: 20.00% (20% improvement)
# 
# BUSINESS IMPACT (if rolled out):
#    Additional Conversions: XXX
#    Incremental Revenue: $X,XXX

print("TODO: Calculate and interpret business metrics")

## 7. Power Analysis

### What is Power Analysis?

**Definition:** Power analysis estimates the **probability that your test will detect a real effect**, given your sample size and the actual effect size.

**Key concept:**
$$\text{Power} = P(\text{detect effect} \mid \text{effect actually exists})$$

**Power vs. Type I/II errors:**

| Concept | Definition | Risk |
|---------|-----------|------|
| **Type I Error (α)** | Reject H₀ when it's true (false positive) | Claim success when nothing changed |
| **Type II Error (β)** | Fail to reject H₀ when it's false (false negative) | Miss real improvements |
| **Power (1-β)** | Correctly detect effect when it exists | Industry standard: 80% (1 - 0.20) |

**How it works:**
1. **Retrospective power:** Given observed effect size and sample size, what's our power?
   - Did we have enough statistical power to detect this effect?
   
2. **Prospective power:** Given desired power (80%) and expected effect size, how many samples needed?
   - How many users per group to reliably detect the improvement?

**Example:**
```
Observed results:
  Cohen's d = 0.15 (small effect)
  Sample size per group = 5,000
  
Power Analysis:
  Achieved Power = 0.62 (62%)
  
Interpretation: Even with 5,000 users per group, we only have 
a 62% chance of detecting this effect. This is BELOW the 80% 
industry standard! We'd need ~10,000 users per group for 80% power.
```

**Why it's useful for A/B testing:**

| Phase | Power Analysis Use |
|-------|-------------------|
| **Before test (Planning)** | How many users do we need? |
| **After test (Analysis)** | Did we have enough power to trust the results? |
| **Interpreting null results** | Did we fail to detect effect (underpowered) or is there no effect? |

**Real-world scenario:**
```
Test Result: p-value = 0.08 (NOT significant at α=0.05)

With low power (40%):
  → "Maybe there IS an effect, but we missed it (Type II error)"
  → Need to run test longer or change approach

With high power (92%):
  → "We had high chance to detect effect, but didn't"
  → Likely NO real effect exists; safe to declare failure
```

---

### How Power Analysis Guides Our Decision

**Our workflow:**

1. **Set up experiment** (before running):
   - Expected effect size: Cohen's d = 0.15 (conservative estimate)
   - Desired power: 80% (industry standard)
   - Significance level: α = 0.05
   - Question: "How many users per group do we need?"
   - Answer: Use `tt_ind_solve_power()` to calculate required sample size

2. **Run experiment** (collect data):
   - We simulated 5,000 users per group

3. **Analyze results** (after experiment):
   - Calculate observed effect size from actual data
   - Calculate achieved power with 5,000 users per group
   - Interpret: "Did our 5,000 users per group have enough power?"
   - Question: "Should we trust our p-value = 0.08?"

**Decision framework:**

| Scenario | Achieved Power | P-value | Conclusion |
|----------|----------------|---------|-----------|
| High power (92%) | High | 0.03 | ✅ REJECT H₀: Real effect detected |
| High power (92%) | High | 0.08 | ✅ FAIL TO REJECT H₀: No effect likely exists |
| Low power (35%) | Low | 0.08 | ❓ UNCLEAR: May have missed real effect (Type II error) |
| Low power (35%) | Low | 0.03 | ✅ REJECT H₀: Effect must be real (we almost missed it!) |

**Why this matters:**
- **High power + p > 0.05** = Safe to say treatment doesn't work
- **Low power + p > 0.05** = Can't tell if treatment works or sample was too small
- **High power + p < 0.05** = Confident treatment works
- **Low power + p < 0.05** = Effect exists but we got lucky (fragile result)

---

In [ ]:
# TODO: Power Analysis - calculate statistical power
# Requirements:
# 1. Calculate Cohen's d effect size from observed AOV difference
# 2. Use tt_ind_solve_power() to calculate achieved power:
#    - effect_size: Cohen's d
#    - nobs1: sample size per group
#    - alpha: 0.05
#    - alternative: 'two-sided'
#
# 3. Interpret power:
#    - Power ≥ 0.80: Adequate (80%+ chance of detecting effect if real)
#    - Power < 0.80: Low (need more samples)
#
# 4. Calculate required sample size for 80% power with observed effect size:
#    - Use tt_ind_solve_power() with power=0.80, solving for nobs1
#
# Expected output:
# Observed Effect Size (Cohen's d): 0.XXXX
# Sample Size (per group): 5,000
# Achieved Power: 0.XXXX (XX%)
# 
# Interpretation: [adequate/insufficient]
# Sample size needed for 80% power: X,XXX per group

print("TODO: Perform power analysis")

## 8. Visualization

In [ ]:
# TODO: Create visualization dashboard with 4 subplots
# Requirements:
# 1. Subplot 1: Bar chart comparing conversion rates
#    - X-axis: Control vs Treatment
#    - Y-axis: Conversion Rate (%)
#    - Add value labels on bars
#
# 2. Subplot 2: Bar chart comparing AOV
#    - X-axis: Control vs Treatment
#    - Y-axis: Average Order Value ($)
#    - Add value labels on bars
#
# 3. Subplot 3: Histogram of AOV distributions
#    - Overlay Control and Treatment distributions
#    - Use alpha transparency for overlap
#    - Add legend
#
# 4. Subplot 4: Text summary of test results
#    - Z-statistic and p-value
#    - T-statistic and p-value
#    - Cohen's d
#    - Significance indicators (✓ or ✗)

# fig, axes = plt.subplots(2, 2, figsize=(14, 10))
# TODO: Implement plotting logic
# TODO: Add titles, labels, and formatting
# TODO: Display figure

print("TODO: Create visualization dashboard")

## 9. Summary Report

In [ ]:
# TODO: Generate comprehensive A/B test report
# Requirements:
# 1. Create a dictionary (or nested dict) with sections:
#    - experiment_name, test_period, hypothesis
#    - sample_sizes (control, treatment, total)
#    - conversion_metrics (rates, lift, z_stat, p_value, significant flag)
#    - aov_metrics (means, lift, t_stat, p_value, cohens_d, significant flag)
#    - power_analysis (achieved_power, required_n)
#    - business_impact (incremental_conversions, incremental_revenue)
#
# 2. Display report in JSON format for clarity
#
# 3. Create decision framework:
#    - IF conversion_significant OR aov_significant:
#        → "SHIP IT: Results are statistically significant"
#    - ELSE:
#        → "HOLD: Results are NOT statistically significant"
#
# 4. For SHIP decision, display:
#    - p-values with significance indicators
#    - Projected incremental revenue
#
# Expected decision output:
# ✓ SHIP IT or ✗ HOLD
# Reasoning with p-values and business impact

print("TODO: Generate final A/B testing report and recommendation")

## 10. Production Implementation

### Key Functions to Migrate to `src/analysis/ab_test.py`

After validating the code above, copy the following implementations to `src/analysis/ab_test.py`:

1. **`ABTestAnalyzer.calculate_conversion_lift()`**
   - Calculate control & treatment conversion rates
   - Return absolute and relative lift

2. **`ABTestAnalyzer.run_z_test()`**
   - Use `proportions_ztest()` from statsmodels
   - Return (p_value, is_significant, z_statistic)

3. **`ABTestAnalyzer.run_t_test()`**
   - Use `scipy.stats.ttest_ind()`
   - Return (p_value, is_significant, t_statistic, cohens_d)

4. **`ABTestAnalyzer.calculate_sample_size()`**
   - Use `tt_ind_solve_power()` for sample size calculation
   - Support both retrospective (power given n) and prospective (n given power)

5. **`ABTestAnalyzer.generate_summary_report()`**
   - Aggregate all metrics into a comprehensive report dict
   - Include business impact projection

### Data Flow Integration

```
Data Sources (Kaggle)
    ↓
Hash-based user randomization (get_group())
    ↓
Effect injection (treatment uplift)
    ↓
Statistical tests (Z-test, T-test)
    ↓
Streamlit dashboard (Stage 4)
    ↓
Business decision (ship or hold)
```



## 📌 Implementation Task: Complete A/B Testing Functions

**After testing all statistical analyses in this notebook, implement in:** `src/analysis/ab_test.py`

### Methods to Implement in `ABTestAnalysis` Class:

1. **`get_group(user_id)`** - Deterministic user assignment
   - Use MD5 hash to assign users to Control/Treatment
   - Ensures same user always sees same variant

2. **`calculate_descriptive_stats(ab_test_df)`**
   - Input: DataFrame with user_id, variant_group, is_converted, order_value
   - Output: Dict with counts, conversion rates, AOV for each group
   - Calculate: total_users, conversions, conv_rate, mean_aov, std_aov per group

3. **`perform_z_test(control_conversions, control_n, treatment_conversions, treatment_n)`**
   - Two-proportion Z-test for conversion rates
   - Use `proportions_ztest()` from statsmodels
   - Return: z_stat, p_value, is_significant

4. **`perform_t_test(control_aov, treatment_aov)`**
   - Independent samples t-test for AOV
   - Filter only converted users
   - Calculate Cohen's d effect size
   - Return: t_stat, p_value, cohens_d, is_significant

5. **`calculate_lift(control_metric, treatment_metric)`**
   - Return: Dict with absolute_lift and relative_lift_pct
   - Example: {absolute: 0.02, relative: 20.0}

6. **`project_business_impact(n_treatment, conversion_lift, aov_lift)`**
   - Calculate incremental conversions and revenue
   - Return: Dict with additional_conversions, incremental_revenue

**Test all methods in this notebook, then move to src/analysis/ab_test.py**